# fairness_training — Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fairness_training/fairness_training/blob/main/notebooks/quickstart.ipynb)

This notebook shows how to train a fairness-constrained neural network with `fairness_training` in about 10 minutes:

1. Install the package
2. Generate synthetic tabular data with a protected attribute
3. Train a `FairModel` with `FairModel.wrap()` around a standard PyTorch network
4. Plot training history and per-group prediction distributions
5. Compare `weighted_avg_fairness_gap` (what the paper's theorem actually bounds) vs `pooled_fairness_gap`

## 1. Install

In [ ]:
# Install the package with training and visualization dependencies
!pip install -q fairness_training[train,viz]

# If you hit an onnxscript conflict with torch, run:
# !pip uninstall onnxscript -y

In [ ]:
import torch
import torch.nn as nn
import numpy as np

from fairness_training import FairModel, FairTrainer, create_stratified_dataloaders
from fairness_training.viz import plot_training_history, plot_group_distributions, plot_fairness_tradeoff

print(f"torch {torch.__version__}")
import fairness_training; print(f"fairness_training {fairness_training.__version__}")

## 2. Synthetic dataset

We generate a binary classification dataset where a protected attribute (column 0, e.g. gender) correlates with the outcome — a common source of model bias.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

N = 2000
INPUT_DIM = 16
PROTECTED_COL = 0  # binary {0, 1}

# Protected attribute: 50/50 split
protected = torch.randint(0, 2, (N, 1)).float()

# Other features
other = torch.randn(N, INPUT_DIM - 1)

# Bias: protected group 1 has higher mean on feature 1 → model learns to discriminate
other[:, 0] += protected[:, 0] * 1.5

X = torch.cat([protected, other], dim=1)          # (N, 16)
y = (other[:, 0] + 0.5 * torch.randn(N) > 0).float()  # binary label

print(f"Dataset: {N} samples, {INPUT_DIM} features")
print(f"Label prevalence: {y.mean():.2f}")
print(f"Protected group 0: n={int((protected==0).sum())}")
print(f"Protected group 1: n={int((protected==1).sum())}")

In [ ]:
# Split into train / val / test
N = len(X)
n_train = int(0.70 * N)
n_val   = int(0.15 * N)

X_train, y_train = X[:n_train],         y[:n_train]
X_val,   y_val   = X[n_train:n_train+n_val], y[n_train:n_train+n_val]
X_test,  y_test  = X[n_train+n_val:],   y[n_train+n_val:]

# Stratified dataloaders maintain balanced group representation per batch
# so the hard fairness constraints always have both groups present.
train_loader, val_loader, test_loader = create_stratified_dataloaders(
    X_train, y_train,
    X_val,   y_val,
    X_test,  y_test,
    protected_attr_idx=PROTECTED_COL,
    batch_size_train=256,
    batch_size_eval=256,
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

## 3. Build a fair model with `FairModel.wrap()`

`FairModel.wrap()` takes any existing `nn.Module` and adds differentiable fairness constraints. The output dimension and prediction bounds are inferred automatically from a dry-run forward pass.

In [ ]:
# Any standard PyTorch network works
backbone = nn.Sequential(
    nn.Linear(INPUT_DIM, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
    nn.Sigmoid(),  # outputs in [0, 1]
)

# Wrap with fairness constraints
model = FairModel.wrap(
    backbone,
    protected_attr_idx=PROTECTED_COL,
    input_dim=INPUT_DIM,
    fairness_tolerance=0.05,   # ε: max allowed group mean prediction gap
    fairness_metric='mean_pred',
)

print(model)

## 4. Train

In [ ]:
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

trainer = FairTrainer(model, criterion, optimizer)

history = trainer.fit(
    train_loader,
    val_loader=val_loader,
    epochs=30,
    verbose=1,
)

## 5. Visualize training history

In [ ]:
fig = plot_training_history(history, title="FairModel Training — mean_pred metric, ε=0.05")

## 6. Evaluate and inspect fairness gaps

In [ ]:
metrics = trainer.evaluate(test_loader, return_predictions=True)

print("\n=== Test Results ===")
print(f"Test loss:                       {metrics['test_loss']:.4f}")
print()
print("Fairness gap measures:")
print(f"  weighted_avg_fairness_gap      {metrics['weighted_avg_fairness_gap']:.4f}")
print(f"  (theorem-bounded quantity = batch-size-weighted time-average of per-batch gaps)")
print()
print(f"  pooled_fairness_gap            {metrics['pooled_fairness_gap']:.4f}")
print(f"  (pooled across all test batches — may differ when group ratios vary)")
print()
print(f"  fairness_tolerance (ε):        {model.fairness_tolerance:.4f}")

### Why two gap measures?

The paper's theorem bounds the **batch-size-weighted time-average** of per-batch fairness gaps:

$$\bar{\Delta} = \frac{1}{N} \sum_t n_t \cdot \Delta_t \leq \varepsilon + \frac{\text{const}}{\sqrt{T}}$$

This is `weighted_avg_fairness_gap` in the output.

The **pooled aggregate gap** (all predictions concatenated, one big group-mean difference) is a different quantity. The two coincide only when the group ratio $n_t^0/n_t^1$ is constant across batches. With stratified dataloaders that ratio is approximately constant, so they should be close here.

## 7. Per-group prediction distributions

In [ ]:
# metrics['protected'] is populated when return_predictions=True
fig = plot_group_distributions(
    metrics['predictions'],
    metrics['protected'],
    attr_names={0: 'Protected attr'},
    title="Test-set prediction distributions after fairness training",
)

## 8. Fairness–accuracy tradeoff across ε values

A common figure in fairness papers: train multiple models with different tolerances and plot loss vs gap.

In [ ]:
epsilons = [0.01, 0.05, 0.10, 0.20]
tradeoff_results = {}

for eps in epsilons:
    bb = nn.Sequential(
        nn.Linear(INPUT_DIM, 64), nn.ReLU(),
        nn.Linear(64, 32), nn.ReLU(),
        nn.Linear(32, 1), nn.Sigmoid(),
    )
    m = FairModel.wrap(
        bb, protected_attr_idx=PROTECTED_COL,
        input_dim=INPUT_DIM, fairness_tolerance=eps,
    )
    t = FairTrainer(m, criterion, torch.optim.Adam(m.parameters(), lr=1e-3))
    t.fit(train_loader, epochs=20, verbose=0)
    tradeoff_results[f"ε={eps}"] = t.evaluate(test_loader)
    print(f"ε={eps}  loss={tradeoff_results[f'ε={eps}']['test_loss']:.4f}  "
          f"gap={tradeoff_results[f'ε={eps}']['weighted_avg_fairness_gap']:.4f}")

In [ ]:
fig = plot_fairness_tradeoff(
    tradeoff_results,
    loss_key='test_loss',
    gap_key='weighted_avg_fairness_gap',
    title='Fairness–Accuracy Tradeoff (mean_pred metric)',
)

## Next steps

- **Custom fairness metric**: subclass `FairnessMetric` and pass it to `FairModel`. Use `validate_metric()` to check DPP compliance before training.
- **Custom network**: pass any `nn.Module` to `FairModel.wrap()` — the prediction bounds are inferred automatically.
- **Multiple protected attributes**: pass a list `protected_attr_idx=[0, 1]`; constraints are enforced per attribute.
- **Checkpoint**: `trainer.save_checkpoint('ckpt.pt')` / `trainer.load_checkpoint('ckpt.pt')`.

See the [documentation](https://github.com/fairness_training/fairness_training) and `src/fairness_training/example_usage.py` for more examples.